# Confocal nuclei counting -- Leica `.lif` files

Counts cell nuclei in confocal images stored in a **Leica LAS X `.lif`** container. A `.lif` holds many *series* (images); each series has several channels. The pixel size (um/px) is read from the file's own metadata, so nothing has to be calibrated by hand.

**Pipeline**: open `.lif` -> take the nuclear-stain channel (max-projection if it is a z-stack) -> light smoothing -> local + global threshold -> watershed to split touching nuclei -> count nuclei, measure their area, density per mm2 -> overlay for visual check -> table per series and summary per condition.

**Dependencies**: `readlif`, `numpy`, `pandas`, `opencv-python-headless`, `scipy`, `scikit-image`, `matplotlib`.

**Status**: first version, counting only (foci are not detected yet). The settings were tuned by eye on three images of one file (about 170-540 nuclei each) and have **not** been validated against a manual count -- check the overlays before trusting the numbers.

## Configuration

Set `mode` to choose what to analyze: **`"single"`** analyzes **one image** (`series_name`) and does **not** export the `.csv`; **`"all"`** analyzes **every image** of the file whose name contains `series_filter` (`None` = every image) and **exports the `.csv`** files. Only the section matching `mode` does anything, so **Run All Cells** is safe.

The series names of your file look like `<condition>/<image>` (e.g. `ctrl-72h/Image059`): the part before the slash is used as the experimental group in the summary.

**Channels**: `nucleus_channel` is the nuclear stain (DAPI/Hoechst). In `20250221.lif` channel 1 is the nuclear stain, channel 0 looks like the foci marker and channel 2 is transmitted light -- this was inferred from the look of the images, the channel names are not stored in the file.

**Length settings are in um**, not pixels, so they keep working if the pixel size changes. If nuclei are merged, lower `min_peak_um`; if single nuclei are split in two, raise it.

**Exclusion filters** (the excluded nuclei are *not counted* but are still outlined on the overlay, in a different colour, so you can see what was left out):
- `max_intensity_ratio` -- nuclei much brighter than the typical nucleus of the same image (condensed chromatin of dead / apoptotic / mitotic cells) -> **red**. There is no sharp separation between condensed nuclei and healthy-but-brighter ones: with the values measured on this file about 1% of the nuclei are above 2.0, 3% above 1.7. Lower the value to exclude more, raise it to exclude fewer.
- `min_area_ratio` / `max_area_ratio` -- nuclei that are too small (debris, fragments of a split nucleus) -> **cyan**, or too large (usually two or more merged nuclei) -> **magenta**, compared with the median nucleus area of the image (bright ones are left out of that median). Big healthy nuclei are common (2x the median is not unusual), so the large limit is deliberately generous.
- Set any of them to `None` to switch that filter off.

In [ ]:
mode = "all"                        # "single" -> analyze ONE image (series_name), no csv exported | "all" -> analyze every image (series_filter) and export the csv

lif_path = "./Data/Confocale/20250221.lif"   # <-- path to the .lif file
series_name = "ctrl-72h/Image059"      # <-- used when mode = "single" (see the table of series below for the names)
series_filter = None                   # <-- used when mode = "all": keep series whose name contains this text (e.g. "72h", "ctrl-24h"); None = all

nucleus_channel = 1                    # channel with the nuclear stain (0-based)
min_area_um2 = 15.0                    # objects smaller than this are not counted as nuclei (debris)
min_peak_um = 4.0                      # minimum distance between nucleus centers for the watershed, in um
smooth_um = 0.4                        # smoothing before thresholding, in um
max_intensity_ratio = 2.0              # exclude nuclei brighter than this multiple of the image's median nucleus intensity (dead / apoptotic / condensed); drawn in red; None = off
min_area_ratio = 0.4                   # exclude nuclei smaller than this fraction of the image's median nucleus area (debris, fragments); drawn in cyan; None = off
max_area_ratio = 2.5                   # exclude nuclei larger than this multiple of the image's median nucleus area (merged nuclei); drawn in magenta; None = off
show_overlays = False                  # mode = "all": also display every overlay in the notebook (they are always saved to outdir)
outdir = "Result_LIF"                  # set to None to skip saving overlays / csv to disk

assert mode in ("single", "all"), "mode must be 'single' or 'all'"

In [ ]:
import os
import re

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import ndimage as ndi
from skimage.filters import gaussian, threshold_otsu, threshold_local
from skimage.measure import regionprops, label
from skimage.morphology import remove_small_objects, binary_opening, disk
from skimage.feature import peak_local_max
from skimage.segmentation import watershed, find_boundaries
from readlif.reader import LifFile

%matplotlib inline

## 1. Opening the `.lif` file

`open_lif()` only reads the index of the file (a few seconds even for a multi-GB `.lif`); the pixel data of a series is read when it is analyzed.

In [ ]:
def open_lif(path):
    """Opens a Leica .lif container and returns {series_name: LifImage}. Pixel data is read lazily."""
    lif = LifFile(path)
    return {im.name: im for im in lif.get_iter_image()}


def describe_series(series):
    """One row per series: channels, z/t size, image size, pixel size."""
    rows = []
    for name, im in series.items():
        rows.append({
            "series": name,
            "channels": im.channels,
            "z": im.nz,
            "t": im.nt,
            "size_px": str(im.dims.x) + "x" + str(im.dims.y),
            "um_per_px": round(1.0 / im.scale[0], 4) if im.scale[0] else None,
            "bit_depth": im.bit_depth[0],
        })
    return pd.DataFrame(rows).set_index("series")


def load_channel(im, channel, t=0):
    """Returns one channel as a 2D float32 array; a z-stack is flattened with a maximum-intensity projection."""
    if channel >= im.channels:
        raise ValueError("channel " + str(channel) + " requested but the series only has " + str(im.channels) + " channels")
    plane = None
    for z in range(max(im.nz, 1)):
        frame = np.asarray(im.get_frame(z=z, t=t, c=channel), dtype=np.float32)
        plane = frame if plane is None else np.maximum(plane, frame)
    return plane


series = open_lif(lif_path)
print(str(len(series)) + " series in " + os.path.basename(lif_path))
display(describe_series(series))

## 2. Nuclei segmentation

The nuclear signal is normalized, lightly smoothed and thresholded with a **local** threshold (so dim nuclei in a bright neighbourhood are not lost) combined with a **global** floor (so background noise is not picked up). Touching nuclei are then split with a watershed on the distance transform.

In [ ]:
def segment_nuclei(img, um_per_px, min_area_um2=15.0, min_peak_um=4.0, smooth_um=0.4, block_um=25.0):
    """
    Segments the nuclei of a 2D fluorescence image.

    Returns (labels, mask): labels is an int image (0 = background, 1..N = nuclei),
    mask is the binary foreground before the watershed split.
    """
    a = np.clip(img / np.percentile(img, 99.8), 0, 1)
    smoothed = gaussian(a, sigma=max(smooth_um / um_per_px, 1.0))

    global_thr = threshold_otsu(smoothed)
    block = int(round(block_um / um_per_px)) | 1
    local_thr = threshold_local(smoothed, block_size=block, method="gaussian", offset=0)
    mask = (smoothed > local_thr) & (smoothed > 0.5 * global_thr)
    mask = ndi.binary_fill_holes(mask)
    mask = binary_opening(mask, disk(max(int(round(1.0 / um_per_px)), 1)))
    mask = remove_small_objects(mask, int(min_area_um2 / um_per_px ** 2))

    dist = ndi.distance_transform_edt(mask)
    dist_smooth = gaussian(dist, sigma=max(1.0 / um_per_px, 1))
    peaks = peak_local_max(dist_smooth, min_distance=max(int(min_peak_um / um_per_px), 1),
                           labels=label(mask), exclude_border=False)
    markers = np.zeros(mask.shape, dtype=bool)
    markers[tuple(peaks.T)] = True
    markers, _ = ndi.label(markers)

    labels = watershed(-dist_smooth, markers, mask=mask)
    return labels, mask

## 3. Analysis of one series

`analyze_series()` loads the nuclear channel, segments it, measures every nucleus (area, mean intensity) and then classifies it with `classify_nuclei()`: **counted**, or excluded as **too bright**, **too small** or **too large** (see the filters in Configuration). Only the counted nuclei go into `n_nuclei`, the median area and the density; the excluded ones are reported in their own columns. Counted nuclei touching the image border are reported separately too (`n_border`, orange) -- a cut nucleus is still a cell, so it is counted.

Overlay colours: **green** counted | **orange** counted, touching the border | **red** too bright | **cyan** too small | **magenta** too large.

In [ ]:
LEGEND = ("green = counted | orange = counted, touches border | red = too bright (excluded) | "
          "cyan = too small (excluded) | magenta = too large (excluded)")


def classify_nuclei(nuclei, max_intensity_ratio=2.0, min_area_ratio=0.4, max_area_ratio=2.5):
    """
    Sets nucleus["status"] for every nucleus:
      "ok"     counted
      "bright"  mean intensity above max_intensity_ratio x the median intensity of the image (dead / apoptotic / condensed)
      "small"   area below min_area_ratio x the median area (debris, fragments)
      "large"   area above max_area_ratio x the median area (merged nuclei)
    The median area is taken over the nuclei that are not "bright". A filter set to None is skipped.
    Excluded nuclei stay in the list so that they can be drawn.
    """
    for n in nuclei:
        n["status"] = "ok"
    if not nuclei:
        return nuclei

    if max_intensity_ratio is not None:
        median_intensity = np.median([n["mean_intensity"] for n in nuclei])
        for n in nuclei:
            if n["mean_intensity"] > max_intensity_ratio * median_intensity:
                n["status"] = "bright"

    reference = [n["area_um2"] for n in nuclei if n["status"] == "ok"]
    if reference:
        median_area = np.median(reference)
        for n in nuclei:
            if n["status"] != "ok":
                continue
            if min_area_ratio is not None and n["area_um2"] < min_area_ratio * median_area:
                n["status"] = "small"
            elif max_area_ratio is not None and n["area_um2"] > max_area_ratio * median_area:
                n["status"] = "large"
    return nuclei


def analyze_series(name, im, nucleus_channel=1, min_area_um2=15.0, min_peak_um=4.0, smooth_um=0.4,
                   max_intensity_ratio=2.0, min_area_ratio=0.4, max_area_ratio=2.5):
    """Counts the nuclei of one series. Returns a dict with the counts, per-nucleus measurements and the arrays used for the overlay."""
    if not im.scale[0]:
        raise ValueError("no spatial calibration stored for series '" + name + "'")
    um_per_px = 1.0 / im.scale[0]

    img = load_channel(im, nucleus_channel)
    labels, mask = segment_nuclei(img, um_per_px, min_area_um2=min_area_um2,
                                  min_peak_um=min_peak_um, smooth_um=smooth_um)

    h, w = labels.shape
    nuclei = []
    for p in regionprops(labels, intensity_image=img):
        r0, c0, r1, c1 = p.bbox
        nuclei.append({
            "label": p.label,
            "area_um2": p.area * um_per_px ** 2,
            "mean_intensity": float(p.mean_intensity),
            "centroid": p.centroid,
            "touches_border": r0 == 0 or c0 == 0 or r1 >= h or c1 >= w,
        })
    classify_nuclei(nuclei, max_intensity_ratio=max_intensity_ratio,
                    min_area_ratio=min_area_ratio, max_area_ratio=max_area_ratio)

    counted = [n for n in nuclei if n["status"] == "ok"]
    areas = np.array([n["area_um2"] for n in counted])
    field_um2 = h * w * um_per_px ** 2
    return {
        "name": name,
        "group": name.split("/")[0],
        "um_per_px": um_per_px,
        "n_detected": len(nuclei),
        "n_nuclei": len(counted),
        "n_border": sum(n["touches_border"] for n in counted),
        "n_bright": sum(n["status"] == "bright" for n in nuclei),
        "n_small": sum(n["status"] == "small" for n in nuclei),
        "n_large": sum(n["status"] == "large" for n in nuclei),
        "median_area_um2": float(np.median(areas)) if len(areas) else float("nan"),
        "field_area_um2": field_um2,
        "nuclei_per_mm2": len(counted) / (field_um2 / 1e6),
        "nuclei": nuclei,
        "image": img,
        "labels": labels,
    }


def make_overlay(result):
    """Grayscale image with the outline of every detected nucleus, coloured by class (see LEGEND)."""
    img = result["image"]
    gray = np.clip(img / np.percentile(img, 99.8), 0, 1)
    rgb = cv2.cvtColor((gray * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)

    edges = find_boundaries(result["labels"], mode="inner")
    edges = cv2.dilate(edges.astype(np.uint8), np.ones((2, 2), np.uint8)).astype(bool)

    classes = {"ok": 1, "border": 2, "bright": 3, "small": 4, "large": 5}
    colors = {1: (0, 255, 0), 2: (0, 165, 255), 3: (0, 0, 255), 4: (255, 255, 0), 5: (255, 0, 255)}
    lut = np.zeros(result["labels"].max() + 1, dtype=np.uint8)
    for n in result["nuclei"]:
        key = "border" if (n["status"] == "ok" and n["touches_border"]) else n["status"]
        lut[n["label"]] = classes[key]
    class_map = lut[result["labels"]]
    for code_, color in colors.items():
        rgb[edges & (class_map == code_)] = color
    return rgb


def show_overlay(rgb, title):
    plt.figure(figsize=(9, 9))
    plt.imshow(cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB))
    plt.title(title + "\n" + LEGEND, fontsize=9)
    plt.axis("off")
    plt.show()


def safe_name(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "__", name)

## 4. Results tables

`condition_columns()` splits the group name (e.g. `100-Rx-72h`) into dose, `Rx` yes/no and time, when the name follows that pattern.

Columns of the table / `.csv`: `n_nuclei` = counted nuclei | `n_border` = of those, touching the border | `median_area_um2` = median area of one counted nucleus | `nuclei_per_mm2` = counted nuclei per mm2 | `field_area_um2` = area of the whole image in um2 | `n_excluded_bright`, `n_excluded_small`, `n_excluded_large` = nuclei left out of the count by each filter.

In [ ]:
def condition_columns(group):
    """Best-effort split of a group name like '100-Rx-72h' into dose / rx / time (None when the pattern does not match)."""
    tokens = group.split("-")
    return {
        "dose": tokens[0] if tokens else None,
        "rx": any(t.lower() == "rx" for t in tokens),
        "time": tokens[-1] if len(tokens) > 1 and re.fullmatch(r"\d+h", tokens[-1].lower()) else None,
    }


def results_table(results):
    """One row per analyzed series."""
    rows = []
    for r in results:
        row = {"series": r["name"], "group": r["group"]}
        row.update(condition_columns(r["group"]))
        row.update({
            "n_nuclei": r["n_nuclei"],
            "n_border": r["n_border"],
            "median_area_um2": round(r["median_area_um2"], 1),
            "nuclei_per_mm2": round(r["nuclei_per_mm2"]),
            "field_area_um2": round(r["field_area_um2"]),
            "n_excluded_bright": r["n_bright"],
            "n_excluded_small": r["n_small"],
            "n_excluded_large": r["n_large"],
        })
        rows.append(row)
    return pd.DataFrame(rows).set_index("series")


def group_summary(table):
    """Nuclei per image for every group: number of images, mean, standard deviation and total."""
    g = table.groupby("group", sort=False)["n_nuclei"]
    return pd.DataFrame({
        "n_images": g.count(),
        "mean_nuclei": g.mean().round(1),
        "sd_nuclei": g.std(ddof=1).round(1),
        "total_nuclei": g.sum(),
    })


def plot_group_summary(table):
    order = list(table["group"].drop_duplicates())
    fig, ax = plt.subplots(figsize=(max(8, 0.7 * len(order)), 5))
    for i, grp in enumerate(order):
        vals = table.loc[table["group"] == grp, "n_nuclei"].values
        ax.bar(i, vals.mean(), yerr=vals.std(ddof=1) if len(vals) > 1 else 0, color="#4C72B0", alpha=0.7, capsize=4)
        ax.scatter(np.full(len(vals), i), vals, color="black", s=14, zorder=3)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=60, ha="right")
    ax.set_ylabel("Nuclei per image")
    ax.set_title("Nuclei per image, by group (bars = mean +/- SD, dots = single images)")
    plt.tight_layout()
    plt.show()

## Run: single image

Uses `series_name` from Configuration. Runs only when `mode = "single"`. Shows the result and saves the overlay PNG in `outdir`; **no `.csv` is exported** in this mode.

In [ ]:
if mode != "single":
    print("Skipped: mode = '" + mode + "'. Set mode = 'single' in Configuration to run this section.")
else:
    if series_name not in series:
        raise KeyError("series '" + series_name + "' not found; pick one of the names in the table above")
    result = analyze_series(series_name, series[series_name], nucleus_channel=nucleus_channel,
                            min_area_um2=min_area_um2, min_peak_um=min_peak_um, smooth_um=smooth_um,
                            max_intensity_ratio=max_intensity_ratio, min_area_ratio=min_area_ratio, max_area_ratio=max_area_ratio)
    overlay = make_overlay(result)

    print(series_name + ": " + str(result["n_nuclei"]) + " nuclei counted (" + str(result["n_border"]) +
          " touching the border) | excluded: " + str(result["n_bright"]) + " too bright, " +
          str(result["n_small"]) + " too small, " + str(result["n_large"]) + " too large")
    print("median area " + format(result["median_area_um2"], ".0f") + " um2 | " +
          format(result["nuclei_per_mm2"], ".0f") + " nuclei/mm2 | image area " +
          format(result["field_area_um2"], ".0f") + " um2 | " + format(result["um_per_px"], ".4f") + " um/px")
    show_overlay(overlay, series_name + " -- " + str(result["n_nuclei"]) + " nuclei counted")

    if outdir:
        os.makedirs(outdir, exist_ok=True)
        cv2.imwrite(os.path.join(outdir, safe_name(series_name) + "_overlay.png"), overlay)
        print("Saved the overlay in " + os.path.abspath(outdir) + " (no csv in single mode)")
    display(results_table([result]))

## Run: all images

Uses `series_filter` from Configuration. Runs only when `mode = "all"`. Saves in `outdir` one overlay PNG per image, `results.csv` (one row per image) and `group_summary.csv` (one row per group); set `show_overlays = True` to also see the overlays in the notebook. An image that fails is skipped with a printed warning.

In [ ]:
if mode != "all":
    print("Skipped: mode = '" + mode + "'. Set mode = 'all' in Configuration to run this section.")
else:
    names = [n for n in series if series_filter is None or series_filter in n]
    if not names:
        raise ValueError("no series name contains '" + str(series_filter) + "'")
    if outdir:
        os.makedirs(outdir, exist_ok=True)

    all_results = []
    for k, name in enumerate(names, 1):
        try:
            res = analyze_series(name, series[name], nucleus_channel=nucleus_channel,
                                 min_area_um2=min_area_um2, min_peak_um=min_peak_um, smooth_um=smooth_um,
                                 max_intensity_ratio=max_intensity_ratio, min_area_ratio=min_area_ratio,
                                 max_area_ratio=max_area_ratio)
        except Exception as exc:
            print("[SKIPPED] " + name + ": " + str(exc))
            continue

        overlay = make_overlay(res)
        if outdir:
            cv2.imwrite(os.path.join(outdir, safe_name(name) + "_overlay.png"), overlay)
        if show_overlays:
            show_overlay(overlay, name + " -- " + str(res["n_nuclei"]) + " nuclei")
        print("[" + str(k) + "/" + str(len(names)) + "] " + name + ": " + str(res["n_nuclei"]) + " nuclei counted (excluded: " +
              str(res["n_bright"]) + " bright, " + str(res["n_small"]) + " small, " + str(res["n_large"]) + " large)")

        del res["image"], res["labels"]   # keep memory small: 60 full-size images would not fit otherwise
        all_results.append(res)

    table = results_table(all_results)
    display(table)
    print()
    display(group_summary(table))
    plot_group_summary(table)

    if outdir:
        table.to_csv(os.path.join(outdir, "results.csv"))
        group_summary(table).to_csv(os.path.join(outdir, "group_summary.csv"))